# RFM Analysis
RFM adalah cara mengelompokkan pelanggan berdasarkan 3 angka sederhana:

* **Recency (R)** — sudah berapa lama sejak pelanggan itu belanja terakhir kali
* **Frequency (F)** — berapa kali pelanggan itu belanja (total transaksi)
* **Monetary (M)** — berapa total uang yang sudah dihabiskan pelanggan itu

Tujuannya: dari 3 angka ini, kita bisa kelompokkan pelanggan jadi kategori seperti “pelanggan setia & bernilai tinggi” vs “pelanggan yang mulai hilang” vs “pelanggan baru”. Ini teknik standar di dunia kerja untuk strategi marketing (misal: siapa yang perlu dikirim promo supaya tidak kabur).

**Insight:**

Analisis RFM terhadap seluruh pelanggan Olist (berbasis `customer_unique_id`) 
menunjukkan karakteristik penting: lebih dari 75% pelanggan hanya pernah 
belanja satu kali (frequency = 1), sehingga segmentasi lebih banyak 
dipengaruhi oleh seberapa baru (recency) dan seberapa besar (monetary) 
transaksi tersebut, bukan oleh frekuensi belanja.

Segmentasi akhir (metode kuartil atas gabungan skor R, F, M) membagi 
pelanggan jadi 4 kelompok dengan proporsi seimbang (~23,339 pelanggan 
per segmen):
- **Pelanggan Terbaik** — recency baru & nilai transaksi tinggi, 
  kandidat prioritas untuk program loyalitas
- **Pelanggan Potensial** — performa cukup baik di 2 dari 3 komponen, 
  berpotensi naik kelas dengan sedikit dorongan (promo re-order)
- **Pelanggan Biasa** — performa rata-rata di semua komponen
- **Pelanggan Berisiko Hilang** — recency lama & nilai transaksi 
  rendah, kandidat untuk kampanye re-engagement

**Keterbatasan:** karena mayoritas pelanggan hanya bertransaksi sekali, 
komponen frequency secara praktis kurang bisa membedakan pelanggan 
secara bermakna dibanding recency dan monetary — ini karakteristik 
alami dataset (marketplace dengan tingkat repeat purchase rendah), 
bukan kelemahan metode analisis.

In [2]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    dbname="Olist Ecommerce Database",
    user="postgres",
    password="admin123",
    port=5432
)

In [3]:
query = """
SELECT 
    c.customer_unique_id,
    o.order_id,
    o.order_purchase_timestamp,
    p.payment_value
FROM olist_orders_dataset o
JOIN olist_customers_dataset c ON o.customer_id = c.customer_id
JOIN olist_order_payments_dataset p ON o.order_id = p.order_id
WHERE o.order_status = 'delivered'
"""

df = pd.read_sql(query, conn)
conn.close()

df.head()

C:\Users\alex7\AppData\Local\Temp\ipykernel_31296\3393445721.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,customer_unique_id,order_id,order_purchase_timestamp,payment_value
0,708ab75d2a007f0564aedd11139c7708,b81ef226f3fe1789b1e8b2acac839d17,2018-04-25 22:01:49,99.33
1,a8b9d3a27068454b1c98cc67d4e31e6f,a9810da82917af2d9aefd1278f1dcfa0,2018-06-26 11:01:38,24.39
2,6f70c0b2f7552832ba46eb57b1c5651e,25e8ea4e93396b6fa0d3dd708e76c1bd,2017-12-12 11:19:55,65.71
3,87695ed086ebd36f20404c82d20fca87,ba78997921bbcdc1373bb41e913ab953,2017-12-06 12:04:06,107.78
4,4291db0da71914754618cd789aebcd56,42fdf880ba16b47b59251dd489d4441a,2018-05-21 13:59:17,128.45


In [4]:
# Tanggal acuan "hari ini" — karena dataset ini historis, kita pakai 
# tanggal transaksi terakhir yang ada di data sebagai acuan "sekarang"
tanggal_acuan = df['order_purchase_timestamp'].max()

rfm = df.groupby('customer_unique_id').agg(
    recency=('order_purchase_timestamp', lambda x: (tanggal_acuan - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('payment_value', 'sum')
).reset_index()

rfm.head()

,customer_unique_id,recency,frequency,monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,111,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,114,1,27.19
2,0000f46a3911fa3c0805444483337064,536,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,320,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,287,1,196.89


In [5]:
rfm[['recency', 'frequency', 'monetary']].describe()

,recency,frequency,monetary
count,93357.000000,93357.000000,93357.000000
mean,236.936673,1.033420,165.198772
std,152.584315,0.209099,226.314579
min,0.000000,1.000000,9.590000
25%,113.000000,1.000000,63.060000
50%,218.000000,1.000000,107.780000
75%,345.000000,1.000000,182.560000
max,694.000000,15.000000,13664.080000


In [6]:
# Recency: makin KECIL angkanya (baru belanja) makin BAIK, 
# jadi urutan skornya dibalik (pakai labels menurun)
rfm['r_score'] = pd.qcut(rfm['recency'], q=4, labels=[4, 3, 2, 1])

# Frequency: makin BESAR angkanya (sering belanja) makin BAIK
# Karena banyak pelanggan nilainya sama (1), dipakai rank dulu supaya 
# tidak error saat dibagi kuartil
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4])

# Monetary: makin BESAR angkanya (banyak belanja) makin BAIK
rfm['monetary_score'] = pd.qcut(rfm['monetary'], q=4, labels=[1, 2, 3, 4])

rfm.head()

,customer_unique_id,recency,frequency,monetary,r_score,f_score,monetary_score
0,0000366f3b9a7992bf8c76cfdf3221e2,111,1,141.90,4,1,3
1,0000b849f77a49e4a4ce2b2a4ca5be3f,114,1,27.19,3,1,1
2,0000f46a3911fa3c0805444483337064,536,1,86.22,1,1,2
3,0000f6ccb0745a6a4b88665a16c9f078,320,1,43.62,2,1,1
4,0004aac84e0df4da2b147fca70cf8255,287,1,196.89,2,1,4


In [8]:
# Gabungkan 3 skor jadi 1 kode, misal "432" artinya r=4, f=3, m=2
rfm['rfm_score'] = (
    rfm['r_score'].astype(str) + 
    rfm['f_score'].astype(str) + 
    rfm['monetary_score'].astype(str)
)

# Jumlahkan 3 skor jadi 1 angka (rentang kemungkinan: 3 sampai 12)
rfm['total_score'] = (
    rfm['r_score'].astype(int) + 
    rfm['f_score'].astype(int) + 
    rfm['monetary_score'].astype(int)
)

# Segmentasi total_score memakai metode kuartil
rfm['segmen'] = pd.qcut(
    rfm['total_score'].rank(method='first'), 
    q=4,
    labels=['Pelanggan Berisiko Hilang', 'Pelanggan Biasa', 'Pelanggan Potensial', 'Pelanggan Terbaik']
)

rfm['segmen'].value_counts()

segmen
Pelanggan Berisiko Hilang    23340
Pelanggan Biasa              23339
Pelanggan Potensial          23339
Pelanggan Terbaik            23339
Name: count, dtype: int64